## Generate Colab Python Code - Part 1 (Setup and Ingestion)

### Subtask:
Generate Python code for Google Colab to import necessary libraries, load the `TL-WR740N_V5_UG.pdf` file, chunk the document, create embeddings, and store them in a Chroma vector store.


**Reasoning**:
The subtask requires generating Python code to import libraries, load a PDF, chunk it, create embeddings, and store them in a Chroma vector store. The necessary libraries are already installed from the previous step, so the next logical step is to write the Python code for these operations.



In [5]:
import os

# Ensure all necessary libraries are installed before importing
# This addresses the ModuleNotFoundError by guaranteeing package availability.
!pip install -q langchain langchain-community langchain-huggingface chromadb pypdf sentence-transformers langchain-text-splitters

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Corrected import path
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Ensure the PDF file is accessible
# For example, if you uploaded it directly to Colab or used !wget, it should be in the current directory
pdf_path = 'TL-WR740N_V5_UG.pdf'

# 3. Instantiate PyPDFLoader and load documents
print(f"Loading document from: {pdf_path}")
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"Loaded {len(docs)} pages.")

# 4. Initialize RecursiveCharacterTextSplitter and split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"Split document into {len(chunks)} chunks.")

# 5. Initialize HuggingFaceEmbeddings
embedding_model_name = 'all-MiniLM-L6-v2'
print(f"Initializing embeddings with model: {embedding_model_name}")
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# 6. Create a Chroma vector store from the document chunks and embeddings
collection_name = 'router_manual'
print(f"Creating Chroma vector store with collection name: {collection_name}")
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name
)
print("Chroma vector store created successfully.")

# Optional: Verify by performing a similarity search
query = "How to setup wireless network?"
results = vectordb.similarity_search(query)
print(f"\nSample similarity search results for '{query}':")
for i, doc in enumerate(results[:2]): # Print top 2 results
    print(f"Result {i+1}:\nContent: {doc.page_content[:200]}...\nSource: {doc.metadata.get('source')} Page: {doc.metadata.get('page')}\n")

Loading document from: TL-WR740N_V5_UG.pdf
Loaded 118 pages.
Split document into 410 chunks.
Initializing embeddings with model: all-MiniLM-L6-v2


KeyboardInterrupt: 

**Reasoning**:
The previous steps successfully set up the Chroma vector store. Now, as per the task requirements, the next step is to create an interactive retrieval loop. This code will allow users to input questions and retrieve relevant documents from the created vector store.



In [7]:
print("\n--- Retrieval Loop ---")
print("Enter your questions. Type 'exit' to quit.\n")

while True:
    query = input("Your question: ")
    if query.lower() == 'exit':
        print("Exiting retrieval loop.")
        break

    # Perform similarity search
    results = vectordb.similarity_search(query, k=6)

    if results:
        print(f"\nFound {len(results)} relevant documents for: '{query}'")
        for i, doc in enumerate(results):
            print(f"\n--- Document {i+1} ---")
            print(f"Content: {doc.page_content.strip()}\n")
            print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")
    else:
        print("No relevant documents found.")
    print("\n--------------------\n")


--- Retrieval Loop ---
Enter your questions. Type 'exit' to quit.

Your question: What are the factory default user name and password?

Found 6 relevant documents for: 'What are the factory default user name and password?'

--- Document 1 ---
Content: • Update the configuration with the file (system will reboot automatically. 
4.17.7 Password 
Choose menu “System Tools → Password”, and then you can change the factory default user 
name and password of the router in the next screen as shown in Figure 4-94. 
91

Source: TL-WR740N_V5_UG.pdf | Page: 100

--- Document 2 ---
Content: Figure 4-91 Restore Factory Default 
Click the Restore button to reset all configuration settings to their default values.  
• The default User Name: admin 
• The default Password: admin 
• The default IP Address: 192.168.0.1 
• The default Subnet Mask: 255.255.255.0 
 Note: 
All changed settings will be lost when defaults are restored. 
4.17.5 Backup & Restore 
Choose menu “System Tools → Backup & Restore ”, 

KeyboardInterrupt: Interrupted by user

In [ ]:
# --- BAGIAN INI UNTUK TESTING INTERAKTIF (Looping) ---

print("\n" + "="*50)
print("🤖 SISTEM SIAP! Silakan tanya tentang Router TP-Link.")
print("Ketik 'exit' atau 'keluar' untuk berhenti.")
print("="*50 + "\n")

while True:
    # 1. Input pertanyaan user
    query = input("\nMasukan Pertanyaan (Inggris): ")

    # Cek jika user ingin keluar
    if query.lower() in ['exit', 'quit', 'keluar', 'stop']:
        print("Terima kasih! Sesi berakhir.")
        break

    if not query.strip(): # Cek jika input kosong
        continue

    print(f"🔍 Sedang mencari jawaban untuk: '{query}'...")

    # 2. Lakukan pencarian (Retrieval) - Ambil 4 dokumen terbaik
    results = vectordb.similarity_search(query, k=4)

    # 3. Tampilkan Hasil dengan Rapi
    print("-" * 50)
    found_answer = False

    for i, doc in enumerate(results):
        # Tampilkan snippet konten (hanya 300 huruf pertama biar tidak kepanjangan)
        content_preview = doc.page_content.replace('\n', ' ')

        print(f"[Dokumen {i+1} - Halaman {doc.metadata.get('page', 'Unknown')}]")
        print(f"...{content_preview[:400]}...") # Ambil 400 karakter
        print("-" * 20)

    print("-" * 50)


🤖 SISTEM SIAP! Silakan tanya tentang Router TP-Link.
Ketik 'exit' atau 'keluar' untuk berhenti.


Masukan Pertanyaan (Inggris): is first password is blank?
🔍 Sedang mencari jawaban untuk: 'is first password is blank?'...
--------------------------------------------------
[Dokumen 1 - Halaman 42]
...password in this blank.  4.6.2 Wireless Security  Choose menu “Wireless→Wireless Security”, and then you can configure the security settings  of your wireless network.  There are five  wireless security modes supported by the router : WPA /WPA2-Personal,  WPA/WPA2-Enterprise and WEP.  33...
--------------------
[Dokumen 2 - Halaman 34]
...fields are case-sensitive.   Auth Server - Enter the authenticating server IP address or host name.  25...
--------------------
[Dokumen 3 - Halaman 70]
...entry with the Host Description is Host_1 and MAC Address is 00-11-22-33-44-AA.   61...
--------------------
[Dokumen 4 - Halaman 101]
...TL-WR740N/TL-WR741ND 150Mbps Wireless N Router      Figure 4-94

is first password is blank?

How do I restore the router to factory default?

What if the WAN LED is off?

how much this wireless Mbps?

what if the DHCP server was disable ?